# DMV Practical 4 – Housing Data Wrangling and Analysis

**Aim:** Load and inspect housing data, remove duplicates, handle missing values, merge neighborhood information, perform grouping and feature engineering, visualize prices, and detect outliers.

The PDF's housing dataset URL is not legible in the scan. Upload the original housing_prices.csv when prompted; expected columns include price, bedrooms, bathrooms, sqft_living, sqft_lot and neighborhood.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/ks.../housing-prices/main/housing_prices.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("Upload the housing_prices.csv dataset used in the practical.")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))

pd.set_option("display.max_columns", None)
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())
display(df.tail())
print("Rows:", df.shape[0], "Columns:", df.shape[1])
print(df.dtypes)
df.info()
display(df.describe())

In [ ]:
# Clean column names, remove duplicates and inspect missing values
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()

missing_table = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum()/len(df)*100).round(2)
})
display(missing_table)

numeric_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    if df[col].isnull().sum():
        df[col] = df[col].fillna(df[col].mode()[0])
print("Total missing values:", df.isnull().sum().sum())

In [ ]:
# Merge neighborhood reference data
if "neighborhood" in df.columns:
    neighborhood_data = pd.DataFrame({
        "neighborhood": ["Greenwood", "Northgate", "Kenmore", "Riverside"],
        "location_type": ["Residential", "Urban", "Suburban", "Residential"]
    })
    df = df.merge(neighborhood_data, on="neighborhood", how="left")
    print("Neighborhood data merged.")
display(df.head())

In [ ]:
# Filtering and feature selection
if "bedrooms" in df.columns:
    display(df[df["bedrooms"] >= 3].head())
if "price" in df.columns:
    display(df[df["price"] > df["price"].median()].head())
if "neighborhood" in df.columns:
    display(df[df["neighborhood"] == "Greenwood"])

selected = [c for c in ["price","bedrooms","bathrooms","sqft_living","sqft_lot","neighborhood"] if c in df.columns]
property_subset = df[selected].copy()
display(property_subset.head())

categorical_cols = df.select_dtypes(include="object").columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print("Original shape:", df.shape, "Encoded shape:", df_encoded.shape)

In [ ]:
# Group-wise summaries
if {"neighborhood","price"}.issubset(df.columns):
    neighborhood_summary = df.groupby("neighborhood")["price"].agg(["count","mean","median","min","max"]).sort_values("mean", ascending=False)
    display(neighborhood_summary)
if {"bedrooms","price"}.issubset(df.columns):
    display(df.groupby("bedrooms")["price"].agg(["count","mean","median","min","max"]))
if {"bathrooms","price"}.issubset(df.columns):
    display(df.groupby("bathrooms")["price"].mean().sort_values(ascending=False))

In [ ]:
# Feature engineering
if {"price","sqft_living"}.issubset(df.columns):
    df["price_per_sqft"] = df["price"] / df["sqft_living"]
    print("Average price per square foot:", df["price_per_sqft"].mean())
if "yr_built" in df.columns:
    df["house_age"] = 2026 - df["yr_built"]
display(df.head())

In [ ]:
# Visualizations
if {"bedrooms","price"}.issubset(df.columns):
    df.groupby("bedrooms")["price"].mean().plot(kind="bar", figsize=(5,3), title="Average House Price by Bedrooms")
    plt.tight_layout(); plt.show()
if "price" in df.columns:
    sns.histplot(df["price"], kde=True); plt.title("Distribution of House Prices"); plt.show()
if {"sqft_living","price"}.issubset(df.columns):
    sns.scatterplot(data=df, x="sqft_living", y="price")
    plt.title("Living Area vs House Price"); plt.show()

# IQR outlier detection and capping
if "price" in df.columns:
    q1, q3 = df["price"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    outliers = df[(df["price"] < lower) | (df["price"] > upper)]
    print("Number of outliers:", len(outliers))
    df["price"] = df["price"].clip(lower, upper)
    sns.boxplot(x=df["price"]); plt.title("House Price After Outlier Handling"); plt.show()

numeric_df = df.select_dtypes(include=np.number)
display(numeric_df.corr())
plt.figure(figsize=(10,7))
sns.heatmap(numeric_df.corr(), cmap="coolwarm")
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

print("Final Dataset Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
# ========== DMV PRACTICAL 4: HOUSING DATA WRANGLING ==========

# AIM: Clean housing data, merge information, analyze house prices,
# create new features, detect outliers and visualize relationships.
# FLOW: Load -> Inspect -> Clean -> Merge -> Filter -> Group ->
#       Feature Engineering -> Visualize -> Handle Outliers

# THEORY:
# Data Wrangling = cleaning and organizing raw data for analysis.
# Data Cleaning = handling missing, duplicate and incorrect data.
# Data Merging = combining datasets using common columns.
# Feature Engineering = creating new useful columns.
# Outlier = unusually high or low value.
# Correlation = relationship between two numerical variables (-1 to +1).

# LIBRARIES:
# pandas(pd) = data handling; numpy(np) = numerical operations.
# matplotlib(plt) = plotting graphs; seaborn(sns) = statistical graphs.

# DATA LOADING:
# try/except = attempts URL loading; handles errors with file upload.
# pd.read_csv(url) = loads CSV dataset.
# files.upload() = uploads file in Google Colab.
# next(iter(uploaded)) = gets first uploaded filename.
# pd.set_option("display.max_columns",None) = displays all columns.
# shape = (rows,columns); head()/tail() = first/last 5 rows.
# dtypes = column data types; info() = dataset information.
# describe() = numerical statistical summary.
# display() = displays formatted tables.

# DATA CLEANING:
# columns.str.strip() = removes extra spaces from column names.
# str.lower() = converts text to lowercase.
# str.replace(" ","_") = replaces spaces with underscores.
# duplicated().sum() = counts duplicate rows.
# drop_duplicates() = removes duplicates; copy() = independent copy.
# isnull().sum() = missing values per column.
# len(df) = total rows; round(2) = rounds to 2 decimal places.
# select_dtypes(include=np.number) = numerical columns.
# select_dtypes(exclude=np.number) = non-numerical columns.
# fillna(median()) = fills missing numbers with median.
# fillna(mode()[0]) = fills missing categories with most common value.
# isnull().sum().sum() = total missing values.

# DATA MERGING:
# pd.DataFrame({...}) = creates neighborhood reference table.
# merge(neighborhood_data,on="neighborhood",how="left")
# = adds matching neighborhood information.
# on = common column; how="left" = keeps all original housing rows.
# Unmatched neighborhoods receive NaN in location_type.

# FILTERING & SELECTION:
# df[df["bedrooms"]>=3] = houses having 3+ bedrooms.
# df[df["price"]>df["price"].median()] = above-median prices.
# df[df["neighborhood"]=="Greenwood"] = Greenwood houses.
# [c for c in [...] if c in df.columns] = existing columns only.
# df[selected].copy() = creates selected-column subset.
# >= = greater/equal; > = greater; == = equal.

# ENCODING:
# select_dtypes(include="object") = selects text columns.
# columns.tolist() = converts column names into list.
# pd.get_dummies(df,columns=categorical_cols,drop_first=True)
# = converts categorical columns into numeric indicator columns.
# drop_first=True = removes one category per column to avoid redundancy.
# df_encoded.shape = dimensions after encoding.

# GROUPING & STATISTICS:
# groupby("neighborhood")["price"] = groups prices by area.
# groupby("bedrooms") = groups houses by number of bedrooms.
# groupby("bathrooms") = groups by number of bathrooms.
# agg(["count","mean","median","min","max"]) = summary statistics.
# count = number; mean = average; median = middle value.
# min = lowest; max = highest.
# sort_values("mean",ascending=False) = highest average first.
# issubset(df.columns) = checks if all required columns exist.

# FEATURE ENGINEERING:
# price_per_sqft = price / sqft_living (price per square foot).
# house_age = 2026 - yr_built (age of house in 2026).
# .mean() = average value.
# NOTE: sqft_living=0 can cause infinite price_per_sqft values.

# VISUALIZATION:
# plot(kind="bar",figsize=(5,3)) = vertical bar graph.
# sns.histplot(df["price"],kde=True) = price distribution.
# kde=True = adds smooth distribution curve.
# sns.scatterplot(data=df,x="sqft_living",y="price")
# = shows relationship between living area and price.
# plt.figure(figsize=(10,7)) = graph size in inches.
# title() = heading; tight_layout() = adjusts spacing.
# show() = displays graph.

# OUTLIER DETECTION (IQR METHOD):
# quantile([0.25,0.75]) = calculates Q1 and Q3.
# Q1 = 25th percentile; Q3 = 75th percentile.
# IQR = Q3 - Q1.
# Lower Limit = Q1 - 1.5*IQR.
# Upper Limit = Q3 + 1.5*IQR.
# | = OR condition; len(outliers) = number of outlier rows.
# clip(lower,upper) = caps extreme prices within limits.
# sns.boxplot(x=df["price"]) = displays price distribution.
# NOTE: Capping changes outlier values; it doesn't remove rows.

# CORRELATION:
# select_dtypes(include=np.number) = numerical data only.
# corr() = calculates correlation matrix (-1 to +1).
# +1 = positive; -1 = negative; 0 = no linear correlation.
# sns.heatmap(...,cmap="coolwarm") = colored correlation matrix.

# FINAL CHECK:
# df.shape = final rows and columns.
# isnull().sum().sum() = remaining missing values.
# duplicated().sum() = remaining duplicate rows.

# ========== 4 IMPORTANT VIVA QUESTIONS ==========

# Q1. What is Data Wrangling?
# Ans: Cleaning, transforming and organizing raw data for analysis.

# Q2. How are missing values and duplicates handled?
# Ans: Median fills missing numbers, mode fills categories,
#      and drop_duplicates() removes duplicate rows.

# Q3. What is Data Merging and Feature Engineering?
# Ans: Merging combines datasets using common columns.
#      Feature Engineering creates new columns, like
#      price_per_sqft and house_age.

# Q4. What is the IQR method for outlier detection?
# Ans: IQR = Q3-Q1. Values outside Q1-1.5*IQR and
#      Q3+1.5*IQR are considered possible outliers.
#      clip() caps them within these limits.

# FINAL VIVA EXPLANATION:
# "We load housing data, inspect columns, remove duplicates,
# fill missing values using median/mode, merge neighborhood data,
# filter and group houses, calculate price per sqft and house age,
# visualize prices using graphs, handle outliers using IQR,
# and study relationships using a correlation heatmap."

# IMPORTANT: The given URL is incomplete, so the original
# housing_prices.csv must be uploaded if loading fails.